# Pickup grid

An experiment, not a tool. One cuboid in the dish, the robot picks it up over and
over at different combinations of two parameters, and the share of successful
pickups is recorded. It exists to check the pickup equation empirically; when the
numbers are in, the notebook has done its job.

**The two axes**

* `pickup_offset_mm` — the tip height **above the bottom of the dish**, not above
  the cuboid. It is `PickingConfig.pickup_offset`, and `pickup_height =
  dish_bottom + pickup_offset` is computed by the config itself.
* `aspirate_flow_rate` — the flow rate while aspirating, µl/s.

The third variable, the size of the cuboid, is the operator's: one size is one run
of this notebook under its own label.

**What counts as a success.** After the lift the dish is photographed. Empty means
the cuboid is in the tip. It is then put back and the dish photographed again —
and that second check is not optional. Without it a cuboid that never came back
(still in the tip, dispensed past the dish) leaves an empty dish on the next
iteration, which is scored a success although there was nothing to pick. So there
are three outcomes and not two: `success`, `miss` and `lost`, and a `lost` stops
the loop and calls the operator.

**What is deliberately absent.** `PickingSession` is not used here: no routine, no
plate, no floaters, no batching, no shape filters. The loop is written in this
notebook on top of `hardware/protocols` and `core/vision/cuboids`. Detection is
`detect_boxes` + `build_cuboid_df` + `add_derived` with **no** `select_pickable`:
there is one cuboid in the dish and nothing to filter, so the largest detection
inside the ROI is the cuboid.

**Statistics.** At ten attempts per cell the standard error of a proportion is
about 0.15 at p ≈ 0.5. That separates 0.2 from 0.8. It does **not** separate 0.5
from 0.7, and no amount of colour on a heat map will change that. The intervals
below are Wilson intervals, because the normal approximation is wrong at n = 10.

Nothing in `core/`, `hardware/`, `workflows/` or `config/` is touched by this
notebook.

## 0. Setup

`bench_setup` fixes `MICROPICK_ROOT` and re-exports the common names. It lives one
directory up, so it is found by walking upwards rather than by assuming where the
kernel was started.

In [ ]:
import csv
import json
import math
import re
import shutil
import subprocess
import sys
from dataclasses import dataclass
from datetime import datetime
from pathlib import Path

import pandas as pd


def _notebooks_dir(start: Path) -> Path:
    """The directory holding bench_setup.py, searching upwards from `start`."""
    for d in (start, *start.parents):
        if (d / "bench_setup.py").is_file():
            return d
    raise RuntimeError(
        f"bench_setup.py not found above {start}; start the kernel inside the "
        f"repository")


sys.path.insert(0, str(_notebooks_dir(Path.cwd())))

from bench_setup import *                          # noqa: E402,F401,F403
from micropick.hardware.protocols import (move_relative, move_to,  # noqa: E402
                                          require_ok, xyz)

pd.set_option("display.width", 200)
print(paths.describe())

## 1. Constants

Everything that decides what this run is. `EXPERIMENT_LABEL` names the results
folder and carries the cuboid size, which is the one variable the operator sets by
hand.

`RETURN_FLOW_RATE` is deliberately not taken from the grid. Dropping the cuboid
back at the grid's flow rate would hit it hydrodynamically before the next
attempt, and the flow rate would then be correlated with itself through the
history of the dish. The return is always the same.

`RETURN_OFFSET_MM` is there for the same reason and does the same job for the
other axis: the cuboid goes back at one fixed height above the dish bottom,
never at the height being tested. Together with the dish centre taught below,
that makes every return identical — same place, same height, same flow — so
nothing about the trip back varies with the parameter under test.

In [ ]:
EXPERIMENT_LABEL   = "dead_400um"        # goes into the results folder name
PICKUP_OFFSETS_MM  = [0.5, 0.6, 0.7, 0.8, 0.9, 1.0, 1.1, 1.2, 1.3, 1.4, 1.5]
FLOW_RATES         = [10, 25, 50, 75, 100]   # ul/s, aspiration. Edit per task.
TRIALS_PER_CELL    = 10
RETURN_FLOW_RATE   = 50.0                # ul/s, putting the cuboid back. NOT from the grid.
RETURN_OFFSET_MM   = 1.0                 # mm above the dish bottom, at the centre. NOT from the grid.
ASPIRATE_VOLUME    = 10.0                # ul, constant
SETTLE_S           = 0.75                # pause before a check frame
RECORD_CLIPS       = False               # lower-camera recording

In [ ]:
# Derived from the constants above, printed so the run is described in one place.
N_CELLS = len(FLOW_RATES) * len(PICKUP_OFFSETS_MM)
CUBOID_SIZE_UM = (lambda m: int(m.group(1)) if m else None)(
    re.search(r"(\d+)\s*u?m", EXPERIMENT_LABEL))

print(f"{len(FLOW_RATES)} flow rates x {len(PICKUP_OFFSETS_MM)} heights "
      f"= {N_CELLS} cells, {TRIALS_PER_CELL} trials each "
      f"= {N_CELLS * TRIALS_PER_CELL} trials at most")
print("cuboid size, read off the label:",
      f"{CUBOID_SIZE_UM} um" if CUBOID_SIZE_UM else
      "not stated — run.json will record only the label")

## 2. Connecting

Profile, robot, cameras, detector. Section 5 of `01_robot_session` has to have run
at some point: this needs a pixel map and a pipette offset in the profile, and a
taught `observe` position.

In [ ]:
PROFILE = "lab_main"

profile = load_profile(PROFILE)
profile.require_calibration()
pmap = PixelMap.from_config(profile.pixel_map)
off = profile.calibration.pipette_offset
tip_offset = np.array([off.dx, off.dy], dtype=float)
observe = np.array(profile.where("observe"), dtype=float)

print("profile:", profile.meta.name, "| schema", profile.meta.schema_version)
print(f"map: degree {pmap.config.degree}, holdout "
      f"{pmap.config.holdout_mean_um or float('nan'):.1f} um")
print(f"pipette offset: ({off.dx:.2f}, {off.dy:.2f}) mm, {off.method}")
print("observe:", tuple(round(v, 2) for v in observe))

In [ ]:
# The picking parameters come from the profile and are edited in place for this
# run. dish_bottom is deliberately not set here: it is measured with the tip at
# the end of this section, in the same pose that gives the dish centre.
cfg = profile.picking

cfg.vol = ASPIRATE_VOLUME
cfg.circle_center = (1296, 972)
cfg.circle_radius = 800
cfg.minimum_distance = 2.0            # here it only grows the ROI mask
cfg.lift_mm = 20.0
cfg.clip_max_frames = 600

print(f"lift {cfg.lift_mm} mm, clip cap {cfg.clip_max_frames} frames, "
      f"ROI r={cfg.circle_radius} px about {cfg.circle_center}")

In [ ]:
openapi = connect_robot()               # creates the run and loads the pipette
print("run:", openapi.run_id, "| pipette:", openapi.pipette_id)
# A tip has to be fitted, and the offset in the profile has to belong to that tip:
# every tip seats differently. Pick one up from 01_robot_session if needed.

In [ ]:
cams = open_cameras(profile)
over_cam = cams.open("overview_cam")
print(over_cam)

problems = profile.pixel_map.check_camera(over_cam.resolution)
require(not problems,
        "the calibration does not match the camera: " + "; ".join(problems))

under_cam = (cams.open("underview_cam", resolution=(2000, 1500))
             if RECORD_CLIPS else None)
if under_cam is not None:
    print(under_cam)

In [ ]:
from ultralytics import YOLO

CUBOID_WEIGHTS = "cuboid_bbox_v4-11_best.pt"
_weights = paths.ml_models_dir() / CUBOID_WEIGHTS
require(_weights.exists(),
        f"cuboid detector weights not found: {_weights}\n"
        f"put the .pt file in {paths.ml_models_dir()} "
        f"(weights are not tracked in the repository)")
cuboid_model = YOLO(str(_weights))
print("loaded", CUBOID_WEIGHTS)

### The dish: its bottom and its centre

Both numbers come out of **one** pose. Jog the tip to the middle of the dish and
bring it down until it just touches the bottom: the z of that pose is
`dish_bottom`, and its x and y are the point every cuboid is returned to.

Arrows or WASD move x and y, `q` and `e` move z, `+` and `-` change the step,
Enter finishes. The window has to have focus, so a stray keystroke in the
notebook cannot drive the robot. Come down in decreasing steps — 0.5, then 0.1 —
and stop at the touch; the number wanted is where the tip meets the glass, not
where it pushes into it.

**Why the return goes to the centre** and not to where the cuboid was picked up
from: a cuboid put back where it came from wanders. Each attempt leaves it a
little further from where it started, and after two hundred attempts it is
against the rim, in a different depth of medium, in a different part of the frame
and at a different local scale of the pixel map. Returning to one taught point
keeps the dish in the state the run started in. It also makes the returns
identical to each other, which is the argument behind `RETURN_FLOW_RATE` and
`RETURN_OFFSET_MM` applied to the third coordinate.

**Teach it once, before the first trial, and not again during the run.**
`dish_bottom` is the origin of the height axis: moving it halfway through leaves
one half of the table in different coordinates from the other, which is why a
resumption whose `dish_bottom` disagrees with `run.json` is refused rather than
carried on with.

In [ ]:
LIMITS = Limits(x=(0, 380), y=(0, 350), z=(0.1, 150))


def jog(title="", camera=None, step=1.0):
    """Manual control from a window, with the camera live in it."""
    ctrl = JogController(openapi, limits=LIMITS, step=step)
    pos = jog_in_window(ctrl, camera or over_cam, title=title)
    print("stopped at", tuple(round(v, 2) for v in pos))
    return pos


def teach_dish(step: float = 0.5) -> np.ndarray:
    """Jog to the middle of the dish, touch the bottom, keep both numbers.

    The pose is stored in the profile as `dish_center`, all three coordinates, so
    a kernel restart does not mean touching the glass again. `cfg.dish_bottom` is
    set from its z but is not written to picking.json here: this is one
    experiment's dish, and the profile's picking parameters belong to the picking
    session. Save it by hand if this dish is the one the rig will keep.
    """
    x, y, z = (float(v) for v in
               jog("centre of the dish, tip on the bottom, then Enter",
                   step=step))
    cfg.dish_bottom = z
    profile.remember("dish_center", (x, y, z))
    # Off the glass straight away: the number is recorded, and nothing good
    # comes of leaving the tip resting on the bottom while this is read.
    move_relative(openapi, "z", 5.0)
    print(f"dish bottom {z:.2f} mm, centre ({x:.2f}, {y:.2f})")
    return np.array([x, y])


def load_dish() -> np.ndarray:
    """The pose taught earlier, out of the profile. No robot motion."""
    x, y, z = profile.where("dish_center")
    cfg.dish_bottom = float(z)
    print(f"dish bottom {z:.2f} mm, centre ({x:.2f}, {y:.2f}) — from the profile")
    return np.array([float(x), float(y)])

In [ ]:
# Teach it. Use load_dish() instead when the dish has not moved since last time.
dish_center = teach_dish()
# dish_center = load_dish()

print(f"pickup height at the lowest offset "
      f"({min(PICKUP_OFFSETS_MM)} mm): "
      f"{cfg.dish_bottom + min(PICKUP_OFFSETS_MM):.2f} mm")
print(f"the cuboid goes back to ({dish_center[0]:.2f}, {dish_center[1]:.2f}) "
      f"at {cfg.dish_bottom + RETURN_OFFSET_MM:.2f} mm, every time")

## 3. Looking at the dish

One frame, one detection pass, the largest object inside the ROI. No
`select_pickable`: there is a single cuboid in the dish, so a shape window can
only ever throw away the one thing being measured.

The gantry pose is read next to the frame the pixel came from, because the pose is
part of the conversion and not a correction applied afterwards.

`park` belongs here for the same reason: every frame of the experiment is taken
from the one observation pose, so that a pixel in one trial means what it meant in
the last.

In [ ]:
def look(rig):
    """One frame -> (frame, detections, gantry_xy). Nothing is filtered out."""
    frame = rig.camera.read_after(time.monotonic())
    gantry = np.array(xyz(rig.robot)[:2])
    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    roi = vision.roi_mask(gray, rig.cfg, rig.one_d_ratio)
    boxes, confs = vision.detect_boxes(rig.detector, frame, rig.cfg)
    df = vision.build_cuboid_df(
        gray, boxes, confs, roi_mask=roi,
        pad=rig.cfg.otsu_pad, open_k=rig.cfg.otsu_open_k,
        bubble_core_r=rig.cfg.bubble_core_r,
        bubble_ring_window=rig.cfg.bubble_ring_window,
        bubble_min_area_px=rig.cfg.bubble_min_area_px)
    df = vision.add_derived(df, rig.size_ratio, rig.one_d_ratio,
                            rig.cfg.circle_center)
    return frame, df, gantry


def largest(df):
    """The biggest detection, or None on an empty dish."""
    if len(df) == 0:
        return None
    return df.loc[df.area.idxmax()]


def park(rig) -> None:
    """Take up the observation pose and let the dish settle.

    Every frame in this experiment is taken from here, so that the pixels of one
    trial mean the same as the pixels of the next.

    The move is skipped when the gantry is already there. Not only for speed: the
    robot declines an absolute move to the height a long tip already sits at
    (DESIGN section 9), and there is no reason to ask it for one.
    """
    if float(np.linalg.norm(np.array(xyz(rig.robot)) - rig.observe)) > 0.2:
        move_to(rig.robot, rig.observe, min_z_height=rig.cfg.dish_bottom,
                force_direct=True)
    time.sleep(rig.settle_s)

## 4. The rig

Everything a trial needs, passed in rather than read from globals, so the same
functions run against mocks in the dry run at the end.

In [ ]:
class TrialError(RuntimeError):
    """The trial could not be carried out at all, as opposed to failing."""


class GridAborted(RuntimeError):
    """The run stopped and wants a person, not a fix in code."""


@dataclass
class Rig:
    """The hardware and the constants one trial works against."""

    robot: object
    camera: object
    detector: object
    cfg: object                      # a PickingConfig
    pmap: object
    offset: np.ndarray               # pipette offset, mm
    observe: np.ndarray              # the observation pose, xyz
    dish_center: np.ndarray          # taught xy the cuboid is returned to
    one_d_ratio: float               # mm per pixel at the dish centre
    size_ratio: float                # mm^2 per pixel^2 there
    settle_s: float = SETTLE_S
    volume: float = ASPIRATE_VOLUME
    return_flow: float = RETURN_FLOW_RATE
    return_offset: float = RETURN_OFFSET_MM
    recorder: object = None          # a hardware.camera.Recorder, or None
    clip_dir: object = None
    clip_crop: tuple = (0, 0)        # origin of what the recorder stores
    clip_size: tuple = (0, 0)
    under_cam: object = None         # the camera the recorder is attached to
    homography: object = None
    keep_clip: object = None         # callable(row) -> bool
    log: object = print


def make_rig(**over) -> Rig:
    """The bench rig: the objects opened above, plus the local scale.

    The scale is taken at the dish centre and nowhere else. The pixel map is a
    degree-3 polynomial, so asked outside its calibrated bounds it extrapolates
    and answers with a plausible wrong number rather than refusing.
    """
    mmpp = float(np.mean(pmap.mm_per_px(*cfg.circle_center)))
    base = dict(robot=openapi, camera=over_cam, detector=cuboid_model, cfg=cfg,
                pmap=pmap, offset=tip_offset, observe=observe,
                dish_center=np.asarray(dish_center, dtype=float)[:2],
                one_d_ratio=mmpp, size_ratio=mmpp * mmpp)
    base.update(over)
    return Rig(**base)

## 5. Results on disk

`outputs/experiments/pickup_grid/<timestamp>_<label>/` holds `trials.csv`,
`trials.xlsx`, `run.json` and `clips/`.

**The CSV is the source of truth, not the xlsx.** A row cannot be appended to an
xlsx; the whole file is rewritten, and an interruption halfway through leaves a
corrupt archive. A CSV is appended a line at a time, so an interrupted run leaves
valid data behind. The xlsx is generated from it, at the end and on demand, so the
operator gets both.

`run.json` is the snapshot of the conditions. Without it the numbers are
unreadable in six months: which profile, which dish bottom, which tip offset,
which weights, which commit.

In [ ]:
RESULTS_ROOT = paths.outputs_dir() / "experiments" / "pickup_grid"

COLUMNS = ["trial_index", "cell_index", "flow_rate", "pickup_offset_mm",
           "replicate", "outcome", "diameter_microns", "area_px", "cX", "cY",
           "target_x", "target_y", "target_z", "timestamp", "clip_path", "note"]

OUTCOMES = ("success", "miss", "lost", "skipped")


def append_row(run_dir, row: dict) -> None:
    """One line onto trials.csv; the header goes in when the file is created."""
    path = Path(run_dir) / "trials.csv"
    fresh = not path.exists()
    with open(path, "a", newline="", encoding="utf-8") as fh:
        writer = csv.DictWriter(fh, fieldnames=COLUMNS, extrasaction="raise")
        if fresh:
            writer.writeheader()
        writer.writerow({k: row.get(k, "") for k in COLUMNS})


def read_trials(run_dir) -> list:
    """Every row written so far, in order. [] when there is no file yet."""
    path = Path(run_dir) / "trials.csv"
    if not path.exists():
        return []
    with open(path, newline="", encoding="utf-8") as fh:
        return list(csv.DictReader(fh))


def write_xlsx(run_dir):
    """Generate trials.xlsx from the CSV. Safe to call at any time."""
    run_dir = Path(run_dir)
    df = pd.read_csv(run_dir / "trials.csv")
    out = run_dir / "trials.xlsx"
    stats = cell_stats(df)
    with pd.ExcelWriter(out, engine="openpyxl") as xl:
        df.to_excel(xl, sheet_name="trials", index=False)
        stats.to_excel(xl, sheet_name="cells", index=False)
        rate_table(stats).to_excel(xl, sheet_name="success_rate")
    return out

In [ ]:
# The summary the xlsx and the analysis section both rest on. It lives here, next
# to the writing, because run_grid builds the xlsx when it finishes and the two
# would otherwise be defined after the cell that needs them.

def wilson(k: int, n: int, z: float = 1.96) -> tuple:
    """Wilson score interval. At n = 10 the normal approximation is wrong: it puts
    bounds outside [0, 1] and gives a zero-width interval at p = 0 or p = 1."""
    if n == 0:
        return (float("nan"), float("nan"))
    p = k / n
    d = 1 + z * z / n
    centre = (p + z * z / (2 * n)) / d
    half = z * math.sqrt(p * (1 - p) / n + z * z / (4 * n * n)) / d
    return max(0.0, centre - half), min(1.0, centre + half)


def cell_stats(df: pd.DataFrame) -> pd.DataFrame:
    """One row per grid cell: counts, rate, and the Wilson interval.

    A `lost` trial is counted and shown but left out of the rate: the cuboid never
    came back, so whether it was picked up says nothing about the parameters.
    """
    rows = []
    for (flow, offset), g in df.groupby(["flow_rate", "pickup_offset_mm"]):
        n_success = int((g.outcome == "success").sum())
        n_miss = int((g.outcome == "miss").sum())
        scored = n_success + n_miss
        low, high = wilson(n_success, scored)
        rows.append(dict(flow_rate=float(flow), pickup_offset_mm=float(offset),
                         skipped=bool((g.outcome == "skipped").any()),
                         n_success=n_success, n_miss=n_miss,
                         n_lost=int((g.outcome == "lost").sum()),
                         n_scored=scored,
                         rate=(n_success / scored if scored else float("nan")),
                         ci_low=low, ci_high=high))
    return (pd.DataFrame(rows)
            .sort_values(["flow_rate", "pickup_offset_mm"])
            .reset_index(drop=True))


def rate_table(stats: pd.DataFrame) -> pd.DataFrame:
    """Success rate, heights down the side, flow rates across. NaN = no data."""
    return stats.pivot(index="pickup_offset_mm", columns="flow_rate",
                       values="rate")

In [ ]:
def repo_commit() -> dict:
    """The commit this ran at, when the repository is available."""
    def git(*args):
        return subprocess.run(["git", "-C", str(paths.root()), *args],
                              capture_output=True, text=True,
                              timeout=5).stdout.strip()
    try:
        return {"commit": git("rev-parse", "HEAD") or None,
                "dirty": bool(git("status", "--porcelain"))}
    except Exception:
        return {"commit": None, "dirty": None}


def run_snapshot(rig, *, label, offsets, flow_rates, trials, weights) -> dict:
    """The conditions of this run, as they go into run.json."""
    import ultralytics

    c = rig.cfg
    po = profile.calibration.pipette_offset
    return {
        "label": label,
        "cuboid_size_um": CUBOID_SIZE_UM,
        "started_at": datetime.now().isoformat(timespec="seconds"),
        "profile": {"name": profile.meta.name,
                    "schema_version": profile.meta.schema_version},
        "dish_bottom": c.dish_bottom,
        "dish_center": [float(v) for v in rig.dish_center],
        "pipette_offset": {"dx": po.dx, "dy": po.dy, "tip_type": po.tip_type,
                           "method": po.method,
                           "measured_at": (po.measured_at.isoformat()
                                           if po.measured_at else None)},
        "volume_ul": rig.volume,
        "return_flow_rate": rig.return_flow,
        "return_offset_mm": rig.return_offset,
        "settle_s": rig.settle_s,
        "lift_mm": c.lift_mm,
        "grid": {"pickup_offsets_mm": [float(o) for o in offsets],
                 "flow_rates": [float(f) for f in flow_rates],
                 "trials_per_cell": int(trials)},
        "detector": {"weights": weights,
                     "ultralytics": ultralytics.__version__,
                     "imgsz": c.yolo_imgsz, "conf": c.yolo_conf,
                     "iou": c.yolo_iou, "max_det": c.yolo_max_det},
        "vision": {"circle_center": list(c.circle_center),
                   "circle_radius": c.circle_radius,
                   "otsu_pad": c.otsu_pad, "otsu_open_k": c.otsu_open_k,
                   "mm_per_px_at_centre": rig.one_d_ratio},
        "pixel_map": {"degree": pmap.config.degree,
                      "holdout_mean_um": pmap.config.holdout_mean_um,
                      "image_size": list(pmap.config.image_size),
                      "fitted_at": (pmap.config.fitted_at.isoformat()
                                    if pmap.config.fitted_at else None)},
        "clips": {"enabled": rig.recorder is not None,
                  "max_frames": c.clip_max_frames},
        "repo": repo_commit(),
    }


# What must not have changed between the start of a run and a resumption of it.
# dish_bottom is in the list because it is the origin of the height axis: shift it
# halfway through and the two halves of the table are in different coordinates.
#
# dish_center is recorded but deliberately not enforced. It is a pose touched by
# hand, so re-teaching it reproduces it to a few hundredths and no further, and
# the difference does not enter any measurement: the pickup point is detected
# afresh every trial, and the centre only says where the cuboid is put back. A
# dish that really moved shows up as a changed dish_bottom, which is enforced.
MUST_MATCH = ["label", "grid", "dish_bottom", "volume_ul", "return_flow_rate",
              "return_offset_mm"]


def check_snapshot(run_dir, snapshot: dict) -> dict:
    """Write run.json, or refuse a resumption whose conditions have changed."""
    path = Path(run_dir) / "run.json"
    if not path.exists():
        path.write_text(json.dumps(snapshot, indent=2) + "\n", encoding="utf-8")
        return snapshot

    stored = json.loads(path.read_text(encoding="utf-8"))
    differences = [f"  {key}: run.json has {stored.get(key)!r}, "
                   f"now {snapshot.get(key)!r}"
                   for key in MUST_MATCH if stored.get(key) != snapshot.get(key)]
    if differences:
        raise GridAborted(
            "this folder was started under different conditions:\n"
            + "\n".join(differences)
            + "\nCarrying on would mix two experiments in one table. Either "
              "restore the values above, or start a new run folder.")
    return stored


def open_run_dir(resume=None):
    """A fresh results folder, or an existing one to be carried on."""
    RESULTS_ROOT.mkdir(parents=True, exist_ok=True)
    if resume:
        run_dir = RESULTS_ROOT / resume
        require(run_dir.is_dir(), f"no such run folder: {run_dir}")
    else:
        run_dir = RESULTS_ROOT / (time.strftime("%Y%m%d_%H%M%S")
                                  + f"_{EXPERIMENT_LABEL}")
        run_dir.mkdir(parents=True, exist_ok=False)
    (run_dir / "clips").mkdir(exist_ok=True)
    return run_dir


def previous_runs(label: str = EXPERIMENT_LABEL) -> list:
    """Folders for this label that already hold trials, oldest first."""
    if not RESULTS_ROOT.is_dir():
        return []
    return sorted(d for d in RESULTS_ROOT.glob(f"*_{label}")
                  if (d / "trials.csv").exists())

## 6. Clips

Off by default, but the plumbing is here in full, on the model of the
lower-camera recording in `01_robot_session`.

A clip cannot be started once a pickup has gone badly: by the time the outcome is
known the filming is over. So **every pickup goes into a ring buffer and the
buffer is thrown away on success**. `Recorder` is that ring buffer already — a
`deque` bounded by `PickingConfig.clip_max_frames` — and it keeps its frames after
`stop()` until the next `start()`, which is what lets the decision wait for the
confirmation frame two moves later.

What gets saved is one function, so the rule can change without touching the loop.

In [ ]:
def should_keep_clip(row: dict) -> bool:
    """Keep failures and losses only, by default."""
    return row["outcome"] in ("miss", "lost")


def clip_view(camera):
    """(origin, size, transform) for what the recorder should store.

    The same three-in-one as workflows/picking._clip_view: the origin subtracted
    from a point drawn on the clip and the crop applied to the frame have to be
    the same thing, and a crop of 1.0 means the whole frame.
    """
    frac = float(getattr(camera, "crop", 1.0))
    w, h = camera.resolution
    if frac >= 1.0:
        return (0, 0), (w, h), None
    x0, y0, side = vision.center_crop_box((h, w), frac)
    return (x0, y0), (side, side), lambda f: vision.center_crop(f, frac)[0]


def attach_recorder(rig, camera, run_dir, homography=None):
    """Give the rig a recorder on `camera`, writing into run_dir/clips.

    `homography` is the profile's `CameraHomography` when there is one. It is
    passed in rather than read off the profile so that the recording can be
    exercised with no profile at all.
    """
    origin, size, transform = clip_view(camera)
    rig.recorder = camera.record(max_frames=rig.cfg.clip_max_frames,
                                 transform=transform)
    rig.under_cam = camera
    rig.clip_dir = Path(run_dir) / "clips"
    rig.clip_dir.mkdir(parents=True, exist_ok=True)
    rig.clip_crop, rig.clip_size = origin, size
    rig.homography = (Homography.from_config(homography)
                      if homography is not None else None)
    if rig.homography is None:
        rig.log("no upper-to-lower homography: the clips record without a box "
                "on the cuboid, which is not an error")
    return rig


def mark_target(rig, uv, gantry) -> None:
    """Box the cuboid on the clip, if the profile can place one.

    Never fatal and never silent: the box is a viewing aid, and every way of not
    getting one says so, because failing quietly here is how it once went missing
    for a whole run of the machine.
    """
    if rig.recorder is None:
        return
    rig.recorder.clear_roi()
    if rig.homography is None:
        return
    try:
        drift = rig.homography.drift_mm(gantry)
        if drift > rig.cfg.homography_drift_warn_mm:
            rig.log(f"    homography was fitted {drift:.1f} mm from this pose; "
                    f"the box is approximate")
        under = rig.homography.over_to_under(
            np.array([uv]), gantry, rig.camera.resolution,
            rig.under_cam.resolution)
        mark = under - np.array(rig.clip_crop, dtype=float)
        w, h = rig.clip_size
        if not (0 <= mark[0, 0] < w and 0 <= mark[0, 1] < h):
            rig.log("    the ROI box falls outside the clip frame; check that "
                    "the homography was fitted on this disc and this camera")
        rig.recorder.mark_rois(mark)
    except Exception as exc:                    # a clip is never critical
        rig.log(f"    no ROI box on the clip: {exc}")


def estimate_clip_bytes(camera, cfg, n_trials: int) -> float:
    """The worst case: every trial fails and every clip is kept.

    The per-frame figure is a rough H.264 estimate and not a measurement — about
    0.1 byte per pixel at these resolutions and this much motion. It is here to
    tell 2 GB from 200 GB before the run starts, not to be accurate.
    """
    _, (w, h), _ = clip_view(camera)
    return n_trials * cfg.clip_max_frames * w * h * 0.1

## 7. Pre-start check

The dish has to show exactly one cuboid. Zero means there is nothing to pick; more
than one means the largest-object rule is choosing one of several, and every
number after that is about an unknown object. Both are refused rather than warned
about.

**The picture comes before the refusal.** What has to be fixed is in the dish, and
whoever fixes it needs to see where: a count on its own says a second object
exists but not that it is a bubble at the rim, or a chip of debris the ROI will
let in the moment the dish is nudged. So the frame is drawn and shown first, with
every detection the model returned in red, the ones that reached the table in
green, and the ROI as a circle — and only then is the run refused.

It also says how long the run will take and, with clips on, how much disk the
worst case needs.

In [ ]:
def show_detections(rig, frame, df, boxes) -> None:
    """The frame with everything that was found drawn on it.

    Red boxes are every detection the model returned, the ROI included or not;
    green contours are the objects that survived into the table. The circle is
    the ROI as `roi_mask` builds it, the working radius grown by the minimum
    spacing, so it is the boundary the table was actually filtered on.
    """
    vis = frame.copy()
    roi_px = (rig.cfg.circle_radius
              + int(rig.cfg.minimum_distance / rig.one_d_ratio))
    overlays.draw_dish(vis, rig.cfg.circle_center, roi_px)
    for x1, y1, x2, y2 in np.asarray(boxes, dtype=float).reshape(-1, 4):
        cv2.rectangle(vis, (int(x1), int(y1)), (int(x2), int(y2)),
                      (0, 0, 255), 2)
    overlays.draw_contours(vis, df, (0, 255, 0), 2)
    for n, (_, r) in enumerate(df.iterrows(), start=1):
        cv2.putText(vis, f"{n}: {r.diameter_microns:.0f} um",
                    (int(r.cX) + 14, int(r.cY) - 14), cv2.FONT_HERSHEY_SIMPLEX,
                    1.1, (0, 255, 0), 2)
    if vis.shape[1] > 1400:
        vis = cv2.resize(vis, (1400, int(1400 * vis.shape[0] / vis.shape[1])))
    show(vis, "red: every detection.  green: inside the ROI.  circle: the ROI")


def preflight(rig, *, offsets, flow_rates, trials) -> dict:
    """Show what the detector sees, then refuse unless it is exactly one cuboid.

    The picture and the table come before the refusal, and deliberately so: the
    reason to stop is nearly always something in the dish, and it cannot be taken
    out by someone who has only been told a count.
    """
    park(rig)
    t0 = time.monotonic()
    frame, df, gantry = look(rig)
    t_detect = time.monotonic() - t0

    # Everything the model returned, whether or not it survived the ROI and the
    # contour step. One extra inference on the frame already in hand: an object
    # just outside the working circle is not a reason to refuse, but it is very
    # much a reason to look, and this runs once.
    boxes, _ = vision.detect_boxes(rig.detector, frame, rig.cfg)

    print(f"detector: {len(boxes)} box(es), {len(df)} of them inside the ROI "
          f"and contoured, {t_detect:.2f} s per frame")
    for n, (_, r) in enumerate(df.iterrows(), start=1):
        print(f"  {n}: ({r.cX:7.1f}, {r.cY:7.1f})  area {r.area:8.0f} px  "
              f"{r.diameter_microns:6.0f} um  conf {r.conf:.2f}")
    if len(boxes) > len(df):
        print(f"  {len(boxes) - len(df)} detection(s) did not reach the table: "
              f"outside the ROI, or no usable contour. They are in red below.")
    show_detections(rig, frame, df, boxes)

    require(len(df) == 1,
            f"the experiment needs exactly one cuboid in the dish, and the "
            f"detector sees {len(df)} inside the ROI ({len(boxes)} in the whole "
            f"frame). "
            + ("Put one in." if len(df) == 0 else
               "Take the others out, or move the ROI so that only one is in it.")
            + " The picture above is the state that was refused.")

    obj = df.iloc[0]
    require(rig.pmap.covers(float(obj.cX), float(obj.cY)),
            "the cuboid is outside the calibrated area of the pixel map")

    cx, cy = rig.dish_center
    print(f"\ndish bottom {rig.cfg.dish_bottom:.2f} mm; the cuboid goes back to "
          f"({cx:.2f}, {cy:.2f}) at "
          f"{rig.cfg.dish_bottom + rig.return_offset:.2f} mm, every trial")

    n_trials = len(offsets) * len(flow_rates) * trials
    # Three detections, three settles and about eight seconds of gantry motion per
    # trial. The motion figure is a rough bench estimate, not a measurement.
    per_trial = 3 * t_detect + 3 * rig.settle_s + 8.0
    print(f"\nat most {n_trials} trials, about {per_trial:.0f} s each "
          f"-> {n_trials * per_trial / 3600:.1f} h "
          f"(less, where early stopping cuts a flow rate short)")

    if rig.recorder is not None:
        worst = estimate_clip_bytes(rig.under_cam, rig.cfg, n_trials)
        free = shutil.disk_usage(str(rig.clip_dir)).free
        print(f"clips: worst case, with every trial kept, about "
              f"{worst / 1e9:.1f} GB; {free / 1e9:.1f} GB free")
        require(free > worst * 1.2,
                f"not enough room for the worst case: {worst / 1e9:.1f} GB "
                f"needed, {free / 1e9:.1f} GB free")

    return {"diameter_microns": float(obj.diameter_microns),
            "area_px": float(obj.area), "n_boxes": int(len(boxes)),
            "t_detect_s": t_detect}

## 8. One trial

`run_trial` returns one row of the table. The order:

1. frame from the top camera, detection, largest object inside the ROI;
2. that pixel into robot coordinates through `pmap`, plus the profile's pipette
   offset;
3. down to `(x, y, dish_bottom + offset)` with `min_z_height`;
4. `aspirate_in_place(volume, flow_rate)` at the grid's flow rate;
5. lift by `lift_mm`, back to the observation pose, `SETTLE_S`;
6. check frame. **Empty means the cuboid was picked up.**
7. to the taught dish centre at `dish_bottom + RETURN_OFFSET_MM`,
   `dispense_in_place(volume, RETURN_FLOW_RATE)`, lift, park, pause;
8. confirmation frame. **The cuboid has to be back.**

The volume is aspirated and returned in every case, a `miss` included: even when
no cuboid came along there is medium in the tip, and it has to go back.

Step 7 is fixed in all three coordinates and in the flow rate, and none of them
is the parameter under test. Putting the cuboid back where it was picked up from
would let it walk across the dish over a few hundred attempts — out of the depth
of medium, the part of the frame and the local pixel scale the run began in — and
dispensing at the height being tested would make the trip back a function of that
height. `DEPOSIT_LIQUID_BACK` in the picking session does return to the pickup
point, which is right there: it is putting a missed cuboid back where it came
from, not keeping a dish still across a grid.

In [ ]:
def run_trial(rig, *, flow_rate, pickup_offset_mm, trial_index, cell_index,
              replicate) -> dict:
    """One pickup, one check, one return, one confirmation. Returns the row."""
    c = rig.cfg
    c.pickup_offset = float(pickup_offset_mm)   # pickup_height follows from it
    z = c.pickup_height
    lift = c.lift_mm

    row = {k: "" for k in COLUMNS}
    row.update(trial_index=trial_index, cell_index=cell_index,
               flow_rate=float(flow_rate),
               pickup_offset_mm=float(pickup_offset_mm), replicate=replicate,
               target_z=round(z, 3),
               timestamp=datetime.now().isoformat(timespec="seconds"))

    # 1. the dish as it is before the pickup
    park(rig)
    _, df, gantry = look(rig)
    obj = largest(df)
    if obj is None:
        raise TrialError(
            "no cuboid in the dish before the pickup. The experiment needs "
            "exactly one; put it back and run the cell again.")
    uv = (float(obj.cX), float(obj.cY))
    row.update(diameter_microns=round(float(obj.diameter_microns), 1),
               area_px=round(float(obj.area), 1),
               cX=round(uv[0], 1), cY=round(uv[1], 1))

    # 2. pixel -> robot, with the pose read next to the frame
    if not rig.pmap.covers(*uv):
        raise TrialError(f"the cuboid is at ({uv[0]:.0f}, {uv[1]:.0f}), outside "
                         f"the calibrated area of the map")
    x, y = np.asarray(rig.pmap.to_robot(uv[0], uv[1], gantry)) + rig.offset
    row.update(target_x=round(float(x), 3), target_y=round(float(y), 3))

    # 3-5. the pickup, into the ring buffer
    mark_target(rig, uv, gantry)
    if rig.recorder is not None:
        rig.recorder.start()
    move_to(rig.robot, (x, y, z + lift), min_z_height=c.dish_bottom,
            force_direct=True)
    move_to(rig.robot, (x, y, z), min_z_height=c.dish_bottom, force_direct=True)
    require_ok(rig.robot.aspirate_in_place(volume=rig.volume,
                                           flow_rate=float(flow_rate)),
               "aspirate")
    move_relative(rig.robot, "z", lift)
    park(rig)

    # 6. the check frame: an empty dish means the cuboid is in the tip
    _, after, _ = look(rig)
    picked = largest(after) is None
    if rig.recorder is not None and rig.recorder.recording:
        rig.recorder.stop()      # the frames stay buffered until the outcome

    # 7. the return: the taught centre, at the fixed height, at the fixed flow.
    # None of the three is the parameter under test, so the trip back is the same
    # every trial and the cuboid does not wander away from where it started.
    cx, cy = rig.dish_center
    zr = c.dish_bottom + rig.return_offset
    move_to(rig.robot, (cx, cy, zr + lift), min_z_height=c.dish_bottom,
            force_direct=True)
    move_to(rig.robot, (cx, cy, zr), min_z_height=c.dish_bottom,
            force_direct=True)
    require_ok(rig.robot.dispense_in_place(volume=rig.volume,
                                           flow_rate=float(rig.return_flow)),
               "dispense back to the dish")
    move_relative(rig.robot, "z", lift)
    park(rig)

    # 8. the confirmation frame: the cuboid has to be back
    _, back, _ = look(rig)
    returned = largest(back) is not None

    if not returned:
        row["outcome"] = "lost"
        row["note"] = ("after the lift the dish was empty" if picked else
                       "after the lift the cuboid was still there")
    else:
        row["outcome"] = "success" if picked else "miss"
    row["clip_path"] = save_clip(rig, row)
    return row


def save_clip(rig, row: dict) -> str:
    """Write the buffered clip if the rule says to keep it. Returns its path."""
    if rig.recorder is None or not rig.recorder.frames:
        return ""
    keep = rig.keep_clip or should_keep_clip
    if not keep(row):
        return ""
    name = f"trial_{int(row['trial_index']):04d}_{row['outcome']}.mp4"
    path = Path(rig.clip_dir) / name
    try:
        rig.recorder.save_async(str(path), color=True)
    except Exception as exc:                    # a clip is never critical
        rig.log(f"    clip not saved: {exc}")
        return ""
    return f"clips/{name}"

## 9. The operator

Two things stop the run and ask for a person: a `lost` cuboid, and zero successes
at the lowest height. The first is answered from a window rather than from
`input()`, so the answer is given with the dish on screen. The window must have
focus for the keys to arrive.

In [ ]:
def operator_gate(camera, message: str, *, window: str = "pickup grid") -> bool:
    """Show the dish live and wait. Space or r carries on, Esc stops the run."""
    print(f"\n!! {message}")
    print("   space / r = carry on,  Esc = stop the run")
    lines = ["OPERATOR",
             *[message[i:i + 58] for i in range(0, len(message), 58)],
             "space/r = carry on   Esc = stop"]
    view = FrameWindow(window)
    try:
        while True:
            ok, frame = camera.read()
            if not ok:
                cv2.waitKey(20)
                continue
            vis = frame.copy()
            for i, line in enumerate(lines):
                cv2.putText(vis, line, (30, 70 + i * 55),
                            cv2.FONT_HERSHEY_SIMPLEX, 1.4, (0, 0, 255), 3)
            view.show(vis)
            key = cv2.waitKey(20) & 0xFF
            if key in (ord("r"), ord(" ")):
                return True
            if key == 27:
                return False
    finally:
        view.close()

## 10. The grid

The outer loop is over flow rates, the inner one over heights **in increasing
order**. The order is not shuffled: early stopping needs an ordered traversal, and
that is a trade made on purpose.

**Early stopping.** If a flow rate scores zero successes at some height, the
remaining, higher heights for that flow rate are skipped. The next flow rate is
unaffected and starts again from the lowest height.

**Skipped cells go into the table** with the status `skipped` and no trials. A
missing row would mean "not tested" and "the data were lost" at the same time.

**Early stopping does not fire at the first height.** 0.5 is a known working point
from the previous work, so a failure there means the calibration is wrong and not
that the range has ended. That stops the run and calls the operator.

**A `lost` trial** is written with the status `lost`, is left out of the success
rate, and stops the loop until the operator confirms. It does consume one of the
`TRIALS_PER_CELL` replicates: it is an attempt that happened, and numbering it
otherwise would make the table disagree with its own row count. The success rate
is therefore `success / (success + miss)` and can rest on fewer than ten attempts,
which the analysis shows as `n` on every cell.

**Carrying on** starts from the first unfinished cell, and inside it from the
replicates already recorded. The grid and the dish bottom in `run.json` have to
match the constants at the top; a disagreement is refused rather than quietly
continued.

In [ ]:
def cell_key(flow, offset) -> tuple:
    return (round(float(flow), 6), round(float(offset), 6))


def read_progress(run_dir) -> dict:
    """Rows already recorded, keyed by (flow rate, height)."""
    done: dict = {}
    for row in read_trials(run_dir):
        if not row.get("flow_rate"):
            continue
        done.setdefault(cell_key(row["flow_rate"], row["pickup_offset_mm"]),
                        []).append(row)
    return done


def next_trial_index(run_dir) -> int:
    indices = [int(r["trial_index"]) for r in read_trials(run_dir)
               if r.get("trial_index")]
    return max(indices) + 1 if indices else 1


def n_successes(rows) -> int:
    return sum(1 for r in rows if r["outcome"] == "success")


def n_scored(rows) -> int:
    return sum(1 for r in rows if r["outcome"] in ("success", "miss"))


def describe_progress(run_dir, *, offsets, flow_rates, trials) -> dict:
    """What is in the folder, and where a run would carry on from."""
    done = read_progress(run_dir)
    rows = read_trials(run_dir)
    if not rows:
        print(f"{Path(run_dir).name}: empty, this is a fresh start")
        return {"resuming": False}

    counts = {o: sum(1 for r in rows if r["outcome"] == o) for o in OUTCOMES}
    print(f"{Path(run_dir).name}: {len(rows)} rows — "
          + ", ".join(f"{k} {v}" for k, v in counts.items() if v))

    for flow in flow_rates:
        for offset in sorted(offsets):
            cell = done.get(cell_key(flow, offset), [])
            if len(cell) < trials and not any(r["outcome"] == "skipped"
                                              for r in cell):
                print(f"  would carry on at flow {flow} ul/s, height {offset} "
                      f"mm, replicate {len(cell) + 1} of {trials}")
                return {"resuming": True, "flow": flow, "offset": offset,
                        "replicate": len(cell) + 1}
    print("  every cell is complete; there is nothing to carry on with")
    return {"resuming": True, "complete": True}

In [ ]:
def run_grid(rig, run_dir, *, offsets, flow_rates, trials, gate=None,
             confirmed: bool = False, log=print):
    """Walk the grid, writing one row per trial. Returns the results folder."""
    run_dir = Path(run_dir)
    offsets = sorted(float(o) for o in offsets)
    gate = gate or (lambda message: operator_gate(rig.camera, message))
    done = read_progress(run_dir)

    if done and not confirmed:
        raise GridAborted(
            f"{run_dir.name} already holds trials. Read what describe_progress() "
            f"printed, check that this is the same dish and the same cuboid, and "
            f"call again with confirmed=True. Carrying on is a deliberate act: "
            f"nothing in software can tell a new cuboid from the old one.")

    trial_index = next_trial_index(run_dir)
    cell_index = 0

    for flow in flow_rates:
        stopped_at = None
        for i, offset in enumerate(offsets):
            cell_index += 1
            rows = list(done.get(cell_key(flow, offset), []))

            if stopped_at is not None:
                if not rows:
                    skipped = {k: "" for k in COLUMNS}
                    skipped.update(
                        cell_index=cell_index, flow_rate=float(flow),
                        pickup_offset_mm=float(offset), replicate=0,
                        outcome="skipped",
                        timestamp=datetime.now().isoformat(timespec="seconds"),
                        note=f"early stop: 0 successes at {stopped_at} mm")
                    append_row(run_dir, skipped)
                log(f"[{flow} ul/s, {offset} mm] skipped")
                continue

            if any(r["outcome"] == "skipped" for r in rows):
                stopped_at = offset            # written by an earlier session
                log(f"[{flow} ul/s, {offset} mm] skipped, from the file")
                continue

            while len(rows) < trials:
                replicate = len(rows) + 1
                row = run_trial(rig, flow_rate=flow, pickup_offset_mm=offset,
                                trial_index=trial_index, cell_index=cell_index,
                                replicate=replicate)
                append_row(run_dir, row)
                rows.append(row)
                trial_index += 1
                log(f"[{flow} ul/s, {offset} mm] {replicate}/{trials} "
                    f"{row['outcome']}"
                    + (f"  ({row['note']})" if row["note"] else ""))
                if row["outcome"] == "lost":
                    if not gate("the cuboid is not in the dish after the "
                                "return. Find it, put it back in the dish, "
                                "then carry on."):
                        raise GridAborted("stopped by the operator after a loss")

            successes, scored = n_successes(rows), n_scored(rows)
            log(f"[{flow} ul/s, {offset} mm] {successes}/{scored} successful"
                + (f", {len(rows) - scored} lost" if scored < len(rows) else ""))

            if successes == 0 and scored > 0:
                if i == 0:
                    raise GridAborted(
                        f"0 successes out of {scored} at the lowest height "
                        f"({offset} mm), which is a known working point. That "
                        f"points at the calibration — the dish bottom, the "
                        f"pipette offset, or the tip — and not at the edge of "
                        f"the range. Stopping. Check the calibration and start a "
                        f"new run folder; do not re-measure the dish bottom into "
                        f"this one.")
                stopped_at = offset
                log(f"  early stop: no successes at {offset} mm, so the higher "
                    f"points for {flow} ul/s are skipped")

    write_xlsx(run_dir)
    log(f"\ndone. {run_dir}")
    return run_dir

### Start, or carry on

`RESUME_FOLDER = None` starts a new folder. To continue an interrupted run, put
its folder name there, read what `describe_progress` says, and set
`CONFIRMED = True`.

In [ ]:
RESUME_FOLDER = None            # e.g. "20260901_141233_dead_400um"
CONFIRMED = False               # True only to carry an existing folder on

for old in previous_runs():
    print("existing folder for this label:", old.name)

run_dir = open_run_dir(RESUME_FOLDER)
rig = make_rig()
if RECORD_CLIPS:
    require(under_cam is not None,
            "RECORD_CLIPS is on but the lower camera is not open; re-run the "
            "camera cell")
    attach_recorder(rig, under_cam, run_dir,
                    homography=profile.calibration.homography)

snapshot = run_snapshot(rig, label=EXPERIMENT_LABEL, offsets=PICKUP_OFFSETS_MM,
                        flow_rates=FLOW_RATES, trials=TRIALS_PER_CELL,
                        weights=CUBOID_WEIGHTS)
check_snapshot(run_dir, snapshot)
print("results:", run_dir)
describe_progress(run_dir, offsets=PICKUP_OFFSETS_MM, flow_rates=FLOW_RATES,
                  trials=TRIALS_PER_CELL)

In [ ]:
info = preflight(rig, offsets=PICKUP_OFFSETS_MM, flow_rates=FLOW_RATES,
                 trials=TRIALS_PER_CELL)

In [ ]:
try:
    run_grid(rig, run_dir, offsets=PICKUP_OFFSETS_MM, flow_rates=FLOW_RATES,
             trials=TRIALS_PER_CELL, confirmed=CONFIRMED)
except GridAborted as exc:
    print(f"\nSTOPPED: {exc}")
finally:
    if rig.recorder is not None:
        rig.under_cam.detach(rig.recorder)
        rig.recorder = None
    openapi.retract_axis("leftZ")

In [ ]:
# The xlsx on demand, from whatever the CSV holds at this moment.
print(write_xlsx(run_dir))

## 11. Analysis

Works on the CSV alone and needs nothing from the run above, so a finished folder
can be read in a fresh kernel.

Ten attempts buy less than they look like they do: the standard error of a
proportion is about 0.15 at p ≈ 0.5. A cell at 0.2 is distinguishable from one at
0.8. A cell at 0.5 is not distinguishable from one at 0.7, whatever the colours
suggest.

In [ ]:
import matplotlib.pyplot as plt

ANALYSE = run_dir           # or RESULTS_ROOT / "20260901_141233_dead_400um"

trials = pd.read_csv(Path(ANALYSE) / "trials.csv")
print(trials["outcome"].value_counts().to_string())
print("\nrun.json:")
print(json.dumps(json.loads((Path(ANALYSE) / "run.json").read_text(
    encoding="utf-8")), indent=2)[:1200])

In [ ]:
# cell_stats and rate_table are defined in section 5, next to the writing.
stats = cell_stats(trials)
table = rate_table(stats)
print(table.round(2).to_string())
stats.round(3)

In [ ]:
# The heat map. A skipped cell is not a zero, so it is grey and labelled: the two
# must not be read off the same colour.
fig, ax = plt.subplots(figsize=(1.4 * len(table.columns) + 3,
                                0.5 * len(table.index) + 3))
cmap = plt.get_cmap("viridis").copy()
cmap.set_bad("0.85")

img = ax.imshow(np.ma.masked_invalid(table.values.astype(float)), cmap=cmap,
                vmin=0, vmax=1, aspect="auto", origin="lower")
ax.set_xticks(range(len(table.columns)), [f"{c:g}" for c in table.columns])
ax.set_yticks(range(len(table.index)), [f"{i:g}" for i in table.index])
ax.set_xlabel("aspirate flow rate, ul/s")
ax.set_ylabel("pickup offset above the dish bottom, mm")
ax.set_title(f"success rate — {Path(ANALYSE).name}")

lookup = stats.set_index(["pickup_offset_mm", "flow_rate"])
for yi, offset in enumerate(table.index):
    for xi, flow in enumerate(table.columns):
        if (offset, flow) not in lookup.index:
            ax.text(xi, yi, "-", ha="center", va="center", color="0.4")
            continue
        row = lookup.loc[(offset, flow)]
        if bool(row.skipped):
            ax.text(xi, yi, "skip", ha="center", va="center", color="0.35",
                    fontsize=9, style="italic")
        elif row.n_scored == 0:
            ax.text(xi, yi, "n=0", ha="center", va="center", color="0.35")
        else:
            ax.text(xi, yi, f"{row.rate:.1f}\nn={int(row.n_scored)}",
                    ha="center", va="center", fontsize=9,
                    color="white" if row.rate < 0.6 else "black")
fig.colorbar(img, ax=ax, label="share of successful pickups")
fig.tight_layout()
plt.show()

In [ ]:
# Success against height, one line per flow rate, with Wilson intervals. The bars
# are wide on purpose: that is what ten attempts per cell actually knows.
fig, ax = plt.subplots(figsize=(9, 5.5))
for flow, g in stats[~stats.skipped].groupby("flow_rate"):
    g = g[g.n_scored > 0].sort_values("pickup_offset_mm")
    if g.empty:
        continue
    err = np.vstack([g.rate - g.ci_low, g.ci_high - g.rate])
    ax.errorbar(g.pickup_offset_mm, g.rate, yerr=err, marker="o", capsize=3,
                label=f"{flow:g} ul/s")
ax.set_xlabel("pickup offset above the dish bottom, mm")
ax.set_ylabel("share of successful pickups")
ax.set_ylim(-0.05, 1.05)
ax.grid(alpha=0.3)
ax.legend(title="aspirate flow")
ax.set_title("success against height, Wilson 95 % intervals")
fig.tight_layout()
plt.show()

In [ ]:
# Did the cuboid itself change over the run? Worth looking at even for a dead one:
# a drift here means the object measured at the end was not the one at the start.
seen = trials[trials.outcome.isin(("success", "miss", "lost"))].dropna(
    subset=["trial_index", "diameter_microns"])

fig, ax = plt.subplots(figsize=(10, 4.5))
for outcome, colour in (("success", "tab:green"), ("miss", "tab:orange"),
                        ("lost", "tab:red")):
    g = seen[seen.outcome == outcome]
    if len(g):
        ax.scatter(g.trial_index, g.diameter_microns, s=18, alpha=0.85,
                   c=colour, label=outcome)
if len(seen) > 1:
    median = seen.diameter_microns.median()
    ax.axhline(median, color="0.5", lw=1)
    ax.text(seen.trial_index.min(), median, f" median {median:.0f} um",
            va="bottom", color="0.35")
ax.set_xlabel("trial index")
ax.set_ylabel("diameter, um")
ax.set_title("the cuboid across the run")
ax.grid(alpha=0.3)
ax.legend()
fig.tight_layout()
plt.show()

print(seen.diameter_microns.describe().round(1).to_string())

## 12. Dry run on mocks

The notebook as a whole cannot run on mocks — it needs a dish, a tip and a
camera — but the parts that decide what is written down can, and those are the
parts that are hard to check on the bench: by the time a status is wrong, the
cuboid has been moved three hundred times.

A mock robot catches the cuboid only below a threshold height, so a small grid
gives both successes and misses; one return "loses" the cuboid on purpose, so the
`lost` branch and the operator gate are exercised too. It is a check of the
bookkeeping, not of the physics.

The grid here is three heights rather than two: with two, a failure at the first
height stops the run instead of skipping anything, so no `skipped` row could ever
be produced and the very thing to be checked would be missing.

This lives in the notebook and not in `tests/`: it belongs to an experiment, not
to the package.

In [ ]:
# --- Dry run on mocks -------------------------------------------------------
from micropick.config.schema import PickingConfig
from micropick.config.schema import PixelMap as PixelMapConfig
from micropick.hardware.mock import MockRobot, open_mock_camera

MW, MH = 900, 700                       # mock frame
MK = 0.05                               # mm per pixel of the toy linear map
M_OBSERVE = np.array([150.0, 150.0, 80.0])
M_OFFSET = np.array([1.0, -2.0])        # pipette offset
M_BOTTOM = 66.4
M_CATCH_Z = M_BOTTOM + 0.6              # the tip only catches below this height
M_CENTER = M_OBSERVE[:2] + M_OFFSET     # the taught centre: pixel (MW/2, MH/2)
M_START_UV = (420.0, 330.0)             # where the cuboid begins, off centre


def mock_pixel_map() -> PixelMap:
    """to_robot(u, v, g) = g + MK * (u - cu, v - cv)."""
    s = max(MW, MH) / 2.0
    return PixelMap(PixelMapConfig(
        degree=1, cu=MW / 2, cv=MH / 2, s=s,
        coef=[[0.0, MK * s], [MK * s, 0.0]], zero=[0.0, 0.0],
        ref=[MW / 2, MH / 2], bounds=[0.0, 0.0, float(MW), float(MH)],
        image_size=[MW, MH], sweep_z=0.0))


class MockDish:
    """One cuboid, drawn as a bright square unless it is in the tip or gone."""

    def __init__(self, uv=M_START_UV, half: int = 18):
        self.uv = np.array(uv, dtype=float)
        self.half = half
        self.present = True
        self.held = False

    @property
    def deck(self) -> np.ndarray:
        return (M_OBSERVE[:2] + MK * (self.uv - np.array([MW / 2, MH / 2]))
                + M_OFFSET)

    def put_at(self, deck_xy) -> None:
        """Where a dispense leaves the cuboid: the inverse of `deck`."""
        self.uv = ((np.asarray(deck_xy, dtype=float) - M_OBSERVE[:2] - M_OFFSET)
                   / MK + np.array([MW / 2, MH / 2]))

    def render(self, i):
        frame = np.full((MH, MW, 3), 20, np.uint8)
        if self.present and not self.held:
            u, v = int(self.uv[0]), int(self.uv[1])
            cv2.rectangle(frame, (u - self.half, v - self.half),
                          (u + self.half, v + self.half), (220, 220, 220), -1)
        return frame


class _Arr:
    def __init__(self, a):
        self._a = np.asarray(a, dtype=float)

    def cpu(self):
        return self

    def numpy(self):
        return self._a


class _Boxes:
    def __init__(self, xyxy, conf):
        self.xyxy, self.conf = _Arr(xyxy), _Arr(conf)


class _Result:
    def __init__(self, boxes):
        self.boxes = boxes


class MockYOLO:
    """Boxes the cuboid when it is visible. Not a model: a stand-in for one."""

    def __init__(self, dish: MockDish):
        self.dish = dish

    def predict(self, source, **kw):
        if not (self.dish.present and not self.dish.held):
            return [_Result(_Boxes(np.zeros((0, 4)), np.zeros((0,))))]
        u, v = self.dish.uv
        h = self.dish.half + 2
        return [_Result(_Boxes([[u - h, v - h, u + h, v + h]], [0.9]))]


class MockGridRobot(MockRobot):
    """Catches the cuboid below M_CATCH_Z, and drops it on chosen returns."""

    def __init__(self, dish: MockDish, lose_on=()):
        super().__init__(position=tuple(M_OBSERVE), noise_mm=0.0,
                         speed_mm_s=4000.0)
        self.dish = dish
        self.lose_on = set(lose_on)
        self.returns = 0

    def aspirate_in_place(self, volume, flow_rate, verbose=False):
        super().aspirate_in_place(volume, flow_rate, verbose)
        near = float(np.linalg.norm(self._pos[:2] - self.dish.deck)) < 1.0
        if near and self._pos[2] <= M_CATCH_Z and self.dish.present:
            self.dish.held = True

    def dispense_in_place(self, volume, flow_rate, verbose=False):
        super().dispense_in_place(volume, flow_rate, verbose)
        if self.dish.held:
            self.returns += 1
            self.dish.held = False
            self.dish.put_at(self._pos[:2])     # it lands under the tip
            if self.returns in self.lose_on:
                self.dish.present = False       # never made it back to the dish


def mock_config() -> PickingConfig:
    return PickingConfig(dish_bottom=M_BOTTOM, vol=10.0, lift_mm=5.0,
                         circle_center=(MW // 2, MH // 2), circle_radius=300,
                         minimum_distance=0.0, clip_max_frames=30)


def mock_rig(dish: MockDish, robot, camera) -> Rig:
    cfg_m = mock_config()
    mmpp = MK
    return Rig(robot=robot, camera=camera, detector=MockYOLO(dish), cfg=cfg_m,
               pmap=mock_pixel_map(), offset=M_OFFSET, observe=M_OBSERVE,
               dish_center=M_CENTER, one_d_ratio=mmpp, size_ratio=mmpp * mmpp,
               settle_s=0.02, volume=10.0, return_flow=50.0, return_offset=1.0,
               log=lambda *a: None)

In [ ]:
# --- the checks -------------------------------------------------------------
M_OFFSETS = [0.5, 0.7, 0.9]      # only 0.5 is below M_CATCH_Z
M_FLOWS = [10, 50]
M_TRIALS = 2

dry_dir = RESULTS_ROOT / "_dry_run"
if dry_dir.exists():
    shutil.rmtree(dry_dir)
dry_dir.mkdir(parents=True)
(dry_dir / "clips").mkdir()

dish = MockDish()
robot = MockGridRobot(dish, lose_on={2})        # the second return drops it
camera = open_mock_camera(width=MW, height=MH, fps=120.0, render=dish.render)
rig_m = mock_rig(dish, robot, camera)

gate_calls = []


def mock_gate(message: str) -> bool:
    """The operator finds the cuboid and puts it back."""
    gate_calls.append(message)
    dish.present = True
    return True


M_SNAPSHOT = {"label": "_dry_run", "dish_bottom": M_BOTTOM, "volume_ul": 10.0,
              "return_flow_rate": 50.0, "return_offset_mm": 1.0,
              "grid": {"pickup_offsets_mm": M_OFFSETS, "flow_rates": M_FLOWS,
                       "trials_per_cell": M_TRIALS}}

try:
    check_snapshot(dry_dir, M_SNAPSHOT)
    run_grid(rig_m, dry_dir, offsets=M_OFFSETS, flow_rates=M_FLOWS,
             trials=M_TRIALS, gate=mock_gate, log=lambda *a: None)

    rows = read_trials(dry_dir)
    counts = {o: sum(1 for r in rows if r["outcome"] == o) for o in OUTCOMES}

    ok_written = verdict("the CSV was written line by line",
                         (dry_dir / "trials.csv").exists() and len(rows) == 10,
                         expected="10 rows", got=f"{len(rows)} rows")

    ok_status = verdict(
        "statuses are assigned",
        counts == {"success": 3, "miss": 4, "lost": 1, "skipped": 2},
        expected={"success": 3, "miss": 4, "lost": 1, "skipped": 2},
        got=counts,
        note="0.5 mm is below the catch height and works; 0.7 and 0.9 do not; "
             "one return drops the cuboid")

    ok_gate = verdict("a loss calls the operator", len(gate_calls) == 1,
                      expected="1 call", got=f"{len(gate_calls)} calls")

    stopped = {(float(r["flow_rate"]), float(r["pickup_offset_mm"]))
               for r in rows if r["outcome"] == "skipped"}
    ok_early = verdict(
        "early stopping fires, and only above the failing height",
        stopped == {(10.0, 0.9), (50.0, 0.9)},
        expected="0.9 mm skipped for both flow rates, 0.7 mm tested",
        got=sorted(stopped))

    ok_skipped_rows = verdict(
        "skipped cells are in the table, with no trials",
        all(r["replicate"] == "0" and r["note"].startswith("early stop")
            for r in rows if r["outcome"] == "skipped"),
        expected="replicate 0 and a reason on every skipped row",
        got=[(r["replicate"], r["note"]) for r in rows
             if r["outcome"] == "skipped"])

    at_centre = float(np.hypot(*(dish.uv - np.array([MW / 2, MH / 2]))))
    ok_centre = verdict(
        "the cuboid is put back at the taught centre, not where it came from",
        at_centre < 1.0,
        expected=f"({MW / 2:.0f}, {MH / 2:.0f}) px, the taught centre",
        got=f"({dish.uv[0]:.1f}, {dish.uv[1]:.1f}) px, having started at "
            f"{M_START_UV}")

    # --- carrying on ---------------------------------------------------------
    # Drop everything for the second flow rate, as an interruption would leave
    # it, and run again: the missing cells should come back and nothing already
    # recorded should be repeated.
    kept = [r for r in rows if float(r["flow_rate"]) != 50.0]
    (dry_dir / "trials.csv").unlink()
    for r in kept:
        append_row(dry_dir, r)
    before = {int(r["trial_index"]) for r in kept if r["trial_index"]}

    run_grid(rig_m, dry_dir, offsets=M_OFFSETS, flow_rates=M_FLOWS,
             trials=M_TRIALS, gate=mock_gate, confirmed=True,
             log=lambda *a: None)

    resumed = read_trials(dry_dir)
    indices = [int(r["trial_index"]) for r in resumed if r["trial_index"]]
    ok_resume = verdict(
        "carrying on picks up at the first unfinished cell",
        len(resumed) == 10 and len(indices) == len(set(indices))
        and min(set(indices) - before) > max(before),
        expected="10 rows again, unique trial indices, new ones after the old",
        got=f"{len(resumed)} rows, indices {sorted(indices)}")

    ok_refuse = False
    try:
        check_snapshot(dry_dir, {**M_SNAPSHOT, "dish_bottom": M_BOTTOM + 1.0})
    except GridAborted:
        ok_refuse = True
    verdict("a changed dish bottom is refused, not carried on with", ok_refuse,
            expected="GridAborted", got="accepted" if not ok_refuse else
            "refused")

    xlsx = write_xlsx(dry_dir)
    sheets = pd.ExcelFile(xlsx).sheet_names
    ok_xlsx = verdict("the xlsx is built from the CSV",
                      xlsx.exists() and "trials" in sheets,
                      expected="a trials sheet", got=sheets)

    # --- the clip decision ---------------------------------------------------
    # Whether a clip is kept, not whether it encodes: the codec belongs to the
    # machine. Two trials with a recorder attached, one that works and one that
    # does not, and the buffer has to be dropped for the first and kept for the
    # second.
    attach_recorder(rig_m, camera, dry_dir)     # no homography, so no ROI box
    try:
        won = run_trial(rig_m, flow_rate=10, pickup_offset_mm=0.5,
                        trial_index=901, cell_index=0, replicate=1)
        lost_it = run_trial(rig_m, flow_rate=10, pickup_offset_mm=0.9,
                            trial_index=902, cell_index=0, replicate=1)
        buffered = len(rig_m.recorder.frames)
        verdict("a successful pickup throws its clip away",
                won["outcome"] == "success" and won["clip_path"] == "",
                expected="success, no clip", got=(won["outcome"],
                                                  won["clip_path"] or "none"))
        verdict("a failed pickup keeps its clip, named for the trial",
                lost_it["outcome"] == "miss"
                and lost_it["clip_path"] == "clips/trial_0902_miss.mp4",
                expected="miss, clips/trial_0902_miss.mp4",
                got=(lost_it["outcome"], lost_it["clip_path"] or "none"))
        verdict("the ring buffer stays inside clip_max_frames",
                0 < buffered <= rig_m.cfg.clip_max_frames,
                expected=f"1..{rig_m.cfg.clip_max_frames} frames",
                got=f"{buffered} frames",
                note="encoding is asynchronous and depends on the machine's "
                     "codecs; this checks the decision, not the file")
    finally:
        camera.detach(rig_m.recorder)
        rig_m.recorder = None

    print("\n--- the table the dry run produced ---")
    print(pd.read_csv(dry_dir / "trials.csv")[
        ["trial_index", "flow_rate", "pickup_offset_mm", "replicate",
         "outcome", "note"]].to_string(index=False))
    print("\n--- success rates ---")
    print(rate_table(cell_stats(pd.read_csv(dry_dir / "trials.csv")))
          .round(2).to_string())
finally:
    camera.close()